# Model

## Import Necessary Libraries

In [2]:
import os
import torch
from torch import nn
from torch.utils.data import DataLoader
from torchvision import datasets, transforms

In [6]:
device = torch.accelerator.current_accelerator().type if torch.accelerator.is_available() else "cpu"
print(f"using device: {device}")

using device: cuda


## Define the Class

In [7]:
class NeuralNetwork(nn.Module):
    def __init__(self):
        super().__init__()
        self.flatten = nn.Flatten()
        self.linear_relu_stack = nn.Sequential(
            nn.Linear(28*28, 512),
            nn.ReLU(),
            nn.Linear(512, 512),
            nn.ReLU(),
            nn.Linear(512, 10),
        )

    def forward(self, x):
        x = self.flatten(x)
        logits = self.linear_relu_stack(x)
        return logits

In [8]:
model = NeuralNetwork().to(device)
print(model)

NeuralNetwork(
  (flatten): Flatten(start_dim=1, end_dim=-1)
  (linear_relu_stack): Sequential(
    (0): Linear(in_features=784, out_features=512, bias=True)
    (1): ReLU()
    (2): Linear(in_features=512, out_features=512, bias=True)
    (3): ReLU()
    (4): Linear(in_features=512, out_features=10, bias=True)
  )
)


In [50]:
X = torch.rand(1, 28, 28, device=device)
logits = model(X)
pred_probab = nn.Softmax(dim=1)(logits)
y_pred = pred_probab.argmax(1)
print(f"Predicted class: {y_pred}")

Predicted class: tensor([2], device='cuda:0')


## Model Layers

In [51]:
input_image = torch.rand(3,28,28)
print(input_image.size())

torch.Size([3, 28, 28])


In [52]:
flatten = nn.Flatten()
flat_image = flatten(input_image)
print(flat_image.size())

torch.Size([3, 784])


In [53]:
layer1 = nn.Linear(in_features=28*28, out_features=20)
hidden1 = layer1(flat_image)
print(hidden1.size())

torch.Size([3, 20])


In [54]:
print(f"Before ReLU: {hidden1}\n\n")
hidden1 = nn.ReLU()(hidden1)
print(f"After ReLU: {hidden1}")

Before ReLU: tensor([[ 0.1630, -0.4914,  0.4083,  0.4106, -0.2878,  0.3543, -0.2262, -0.1040,
         -0.1294,  0.1323,  0.0057,  0.3476, -0.1869, -0.0232, -0.2136, -0.0876,
         -0.0199,  0.4138, -0.4667, -0.2257],
        [ 0.2904, -0.4651,  0.3203,  0.2269, -0.2282,  0.1709, -0.3545,  0.0186,
          0.4752,  0.1789, -0.2104,  0.2657, -0.1635, -0.1521, -0.0155, -0.1271,
          0.1938,  0.4860, -0.4264, -0.3234],
        [ 0.2048, -0.1816,  0.1903,  0.2047, -0.0983,  0.1376, -0.5035,  0.1194,
          0.0731, -0.0457,  0.2210,  0.5507, -0.0613,  0.0060,  0.0952, -0.0395,
          0.3063,  0.4198, -0.4100, -0.0488]], grad_fn=<AddmmBackward0>)


After ReLU: tensor([[0.1630, 0.0000, 0.4083, 0.4106, 0.0000, 0.3543, 0.0000, 0.0000, 0.0000,
         0.1323, 0.0057, 0.3476, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.4138,
         0.0000, 0.0000],
        [0.2904, 0.0000, 0.3203, 0.2269, 0.0000, 0.1709, 0.0000, 0.0186, 0.4752,
         0.1789, 0.0000, 0.2657, 0.0000, 0.0000, 0.00

In [ ]:
seq_modules = nn.Sequential(
    flatten,
    layer1,
    nn.ReLU(),
    nn.Linear(20, 10)
)
input_image = torch.rand(3,28,28)
logits = seq_modules(input_image)

softmax = nn.Softmax(dim=1)
print(f"Logits: {logits}\n\n")
pred_probab = softmax(logits)
print(f"Predicted class probabilities: {pred_probab}")

## Model Parameters

In [58]:
print(f"Model structure: {model}\n\n")

for name, param in model.named_parameters():
    print(f"Layer: {name} | Size: {param.size()} | Values : {param[:2]} \n")

Model structure: NeuralNetwork(
  (flatten): Flatten(start_dim=1, end_dim=-1)
  (linear_relu_stack): Sequential(
    (0): Linear(in_features=784, out_features=512, bias=True)
    (1): ReLU()
    (2): Linear(in_features=512, out_features=512, bias=True)
    (3): ReLU()
    (4): Linear(in_features=512, out_features=10, bias=True)
  )
)


Layer: linear_relu_stack.0.weight | Size: torch.Size([512, 784]) | Values : tensor([[ 0.0052,  0.0006, -0.0011,  ...,  0.0041, -0.0287, -0.0003],
        [-0.0168, -0.0247, -0.0298,  ...,  0.0202, -0.0301, -0.0008]],
       device='cuda:0', grad_fn=<SliceBackward0>) 

Layer: linear_relu_stack.0.bias | Size: torch.Size([512]) | Values : tensor([ 0.0305, -0.0178], device='cuda:0', grad_fn=<SliceBackward0>) 

Layer: linear_relu_stack.2.weight | Size: torch.Size([512, 512]) | Values : tensor([[ 0.0227, -0.0326,  0.0118,  ...,  0.0150,  0.0266, -0.0010],
        [ 0.0149, -0.0370, -0.0398,  ...,  0.0129,  0.0356,  0.0057]],
       device='cuda:0', grad_fn=<Sl